# Experimento — Núcleo determinístico (sem LLM)

**Trabalho final de Teoria da Computação — PPCOMP/IFES.**
Verificação de restrições como **monitoramento por autômatos**, aplicada à geração
de consultas Elasticsearch DSL sobre NF-e aninhada.

Este notebook implementa e **demonstra a teoria de forma 100% reprodutível, sem
chamar nenhum LLM e sem acesso à rede**: parser/GLC (monitor gramatical = papel do
VPA), monitor AFD do laço (R1/R2) e interpretador de referência (semântica
`nested` correlacionada × achatada). A geração C0/C2 por LLM real é fase posterior.

Estrutura (células): `00_setup` · `01_dataset` · `02_grammar` · `03_interpreter` ·
`04_run` · `05_metrics` · `06_figuras`.

## `00_setup` — dependências, seed e reprodutibilidade

In [ ]:
# !pip install lark matplotlib
import json, glob, os
import xml.etree.ElementTree as ET  # dados sintéticos locais e confiáveis
import lark
import matplotlib
matplotlib.use("Agg")  # backend headless (reprodutível, sem display)
import matplotlib.pyplot as plt

SEED = 42  # o núcleo determinístico: sem aleatoriedade relevante, sem rede, sem API
print("lark", lark.__version__, "| matplotlib", matplotlib.__version__)
print("Reprodutível: nenhuma chamada a LLM ou rede; saída idêntica a cada execução.")

lark 1.3.1 | matplotlib 3.11.0
Reprodutível: nenhuma chamada a LLM ou rede; saída idêntica a cada execução.


## `01_dataset` — NF-e reais, notas sintéticas T2 e consultas-ouro

Carrega as 10 NF-e reais (cabeçalho plano + itens aninhados) e define notas
multi-item **sintéticas congeladas** com atributos **cruzados** — o cenário que as
notas reais `0007`/`0008` (ambas CFOP 6108) não exibem e que é necessário para o
falso positivo de T2. As consultas-ouro são definidas por nível T0/T1/T2; o
**resultado-ouro** é computado pelo interpretador de referência (célula 04).

In [2]:
NS = {"n": "http://www.portalfiscal.inf.br/nfe"}

def _txt(el, path):
    f = el.find(path, NS)
    return f.text if f is not None else None

def carregar_nfe_reais(pasta):
    docs = []
    for caminho in sorted(glob.glob(os.path.join(pasta, "*.xml"))):
        inf = ET.parse(caminho).getroot().find(".//n:infNFe", NS)
        header = {
            "ide.dhEmi":   _txt(inf, "n:ide/n:dhEmi"),
            "total.vNF":   _txt(inf, "n:total/n:ICMSTot/n:vNF"),
            "total.vICMS": _txt(inf, "n:total/n:ICMSTot/n:vICMS"),
            "emit.xNome":  _txt(inf, "n:emit/n:xNome"),
            "dest.UF":     _txt(inf, "n:dest/n:enderDest/n:UF"),
        }
        itens = [{
            "det.prod.CFOP":          _txt(d, "n:prod/n:CFOP"),
            "det.imposto.ICMS.vICMS": _txt(d, "n:imposto/n:ICMS/n:ICMS00/n:vICMS"),
            "det.imposto.ICMS.pICMS": _txt(d, "n:imposto/n:ICMS/n:ICMS00/n:pICMS"),
        } for d in inf.findall("n:det", NS)]
        docs.append({"id": os.path.basename(caminho).replace("-nfe.xml", ""),
                     "header": header, "itens": itens})
    return docs

def notas_sinteticas_t2():
    """Itens com atributos cruzados: nas notas 'trap', CFOP 6108 está no item de
    ICMS baixo e CFOP 5102 no de ICMS alto — nenhum item satisfaz
    (CFOP=6108 E vICMS>200), mas a consulta achatada casa indevidamente."""
    def item(cfop, vicms):
        return {"det.prod.CFOP": cfop, "det.imposto.ICMS.vICMS": vicms,
                "det.imposto.ICMS.pICMS": "4.0000"}
    def nota(nid, itens):
        return {"id": nid, "itens": itens,
                "header": {"emit.xNome": "Sintetica S/A", "dest.UF": "ES",
                           "total.vNF": "9999.00", "total.vICMS": "100.00",
                           "ide.dhEmi": "2025-03-01T10:00:00-03:00"}}
    return [
        nota("sint_trap_1", [item("6108", "50.00"),  item("5102", "300.00")]),
        nota("sint_trap_2", [item("6108", "120.00"), item("5102", "250.00")]),
        nota("sint_trap_3", [item("5102", "400.00"), item("6108", "10.00")]),
        nota("sint_pos_1",  [item("6108", "260.00"), item("5102", "30.00")]),
        nota("sint_pos_2",  [item("6108", "500.00")]),
        nota("sint_neg_1",  [item("5102", "300.00"), item("5102", "20.00")]),
    ]

CAMPOS_NESTED = {"det"}  # mapping: 'det' é o único campo declarado nested

docs_reais = carregar_nfe_reais("../notas_fiscais")
docs_t2    = notas_sinteticas_t2()

# Consultas-ouro por nível: (pergunta_NL, nivel, dsl, docs_alvo)
def nested(q): return {"nested": {"path": "det", "query": q}}
CONSULTAS = [
    ("Notas com vNF acima de R$ 1.500", "T0",
     {"query": {"range": {"total.vNF": {"gt": 1500}}}}, docs_reais),
    ("Notas com vNF entre R$ 1.500 e R$ 9.000", "T0",
     {"query": {"range": {"total.vNF": {"gte": 1500, "lte": 9000}}}}, docs_reais),
    ("Notas cujo destinatário está no PR", "T0",
     {"query": {"term": {"dest.UF": "PR"}}}, docs_reais),
    ("Notas com algum item de CFOP 6108", "T1",
     {"query": nested({"term": {"det.prod.CFOP": "6108"}})}, docs_reais),
    ("Notas com algum item cujo vICMS é maior que R$ 60", "T1",
     {"query": nested({"range": {"det.imposto.ICMS.vICMS": {"gt": 60}}})}, docs_reais),
    ("Notas com um item que tenha CFOP 6108 e vICMS acima de R$ 200 no mesmo item", "T2",
     {"query": nested({"bool": {"must": [
        {"term": {"det.prod.CFOP": "6108"}},
        {"range": {"det.imposto.ICMS.vICMS": {"gt": 200}}}]}})}, docs_t2),
    ("Notas com um item de CFOP 6108 e vICMS de pelo menos R$ 250 no mesmo item", "T2",
     {"query": nested({"bool": {"must": [
        {"term": {"det.prod.CFOP": "6108"}},
        {"range": {"det.imposto.ICMS.vICMS": {"gte": 250}}}]}})}, docs_t2),
]
print(f"NF-e reais: {len(docs_reais)} (multi-item:",
      sum(1 for d in docs_reais if len(d['itens']) > 1), ")")
print(f"Notas sintéticas T2: {len(docs_t2)} | consultas-ouro: {len(CONSULTAS)}")

NF-e reais: 10 (multi-item: 2 )
Notas sintéticas T2: 6 | consultas-ouro: 7


## `02_grammar` — GLC do subconjunto ES-NF-e e o monitor gramatical

A gramática (em `lark`) espelha a Figura 3.1 do Capítulo 3. A recursão
`clause → nested → clause` gera aninhamento de profundidade arbitrária com
delimitadores balanceados. `monitor_gramatical` é o **decisor de pertinência em
L(G)** — o papel do autômato de pilha visível (VPA) no degrau gramatical.

In [3]:
GRAMATICA_ES = r"""
start    : query
query    : "{" "\"query\"" ":" clause "}"
clause   : bool | term | range | nested
bool     : "{" "\"bool\"" ":" "{" occurs "}" "}"
occurs   : occur ("," occur)*
occur    : OCC ":" "[" clauses "]"
OCC      : "\"must\"" | "\"filter\"" | "\"should\"" | "\"must_not\""
clauses  : clause ("," clause)*
term     : "{" "\"term\"" ":" "{" FIELD ":" value "}" "}"
range    : "{" "\"range\"" ":" "{" FIELD ":" "{" bounds "}" "}" "}"
bounds   : bound ("," bound)*
bound    : BND ":" NUMBER
BND      : "\"gte\"" | "\"gt\"" | "\"lte\"" | "\"lt\""
nested   : "{" "\"nested\"" ":" "{" "\"path\"" ":" PATH "," "\"query\"" ":" clause "}" "}"
value    : FIELD | NUMBER
FIELD    : ESCAPED_STRING
PATH     : ESCAPED_STRING
%import common.ESCAPED_STRING
%import common.NUMBER
%import common.WS
%ignore WS
"""
from lark import Lark, LarkError
_parser = Lark(GRAMATICA_ES, start="start", parser="earley")

def monitor_gramatical(consulta):
    """Decide pertinência em L(G). Aceita dict (AST) ou str (texto JSON)."""
    texto = consulta if isinstance(consulta, str) else json.dumps(consulta)
    try:
        _parser.parse(texto); return True
    except LarkError:
        return False

def checa_aninhamento(texto):
    """Boa-formação de delimitadores balanceados (essência do VPA): {[ empilham,
    ]} desempilham e devem casar; ignora delimitadores dentro de strings."""
    pilha, par = [], {")": "(", "}": "{", "]": "["}
    em_str = esc = False
    for c in texto:
        if em_str:
            if esc: esc = False
            elif c == "\\": esc = True
            elif c == '"': em_str = False
            continue
        if c == '"': em_str = True
        elif c in "{[(": pilha.append(c)
        elif c in ")}]":
            if not pilha or pilha.pop() != par[c]: return False
    return not pilha

def monitor_contextual(consulta, campos_nested):
    """Todo `path` de nested deve apontar para campo declarado nested no mapping."""
    def visita(c):
        if not isinstance(c, dict): return True
        if "nested" in c:
            if c["nested"].get("path") not in campos_nested: return False
            return visita(c["nested"].get("query"))
        if "bool" in c:
            return all(visita(s) for lst in c["bool"].values() for s in lst)
        return True
    return visita(consulta.get("query", consulta))

def consultas_malformadas():
    """REJEITADAS pelo monitor gramatical por violar UMA regra da GLC (delimitadores
    balanceados, para isolar a violação gramatical do balanceamento)."""
    return [
        ('{"query": {"fuzzy": {"campo": "x"}}}', "cláusula inexistente (fuzzy ∉ clause)"),
        ('{"query": {"term": {"a": "1", "b": "2"}}}', "term com dois campos (clause term tem um field:value)"),
        ('{"query": {"bool": {"must": []}}}', "array de occur vazio (clauses exige >= 1)"),
        ('{"query": {"nested": {"query": {"term": {"a": "b"}}}}}', "nested sem path"),
        ('{"range": {"v": {"gt": 1}}}', "falta a raiz \"query\""),
    ]

def cadeias_desbalanceadas():
    """Cadeias com delimitadores NÃO balanceados, para exercitar o checa_aninhamento
    (a essência da pilha visível/VPA): { [ empilham, ] } desempilham e devem casar."""
    return [
        '{"nested": {"path": "det", "query": {"term": {"a": "b"}}',   # faltam 2 fechamentos
        '{"bool": {"must": [ {"term": {"a": "b"}} }}',                # '[' fechado por '}' (mismatch)
        '{"term": {"a": "b"}}}',                                     # um '}' a mais (pop em pilha vazia)
    ]

print("monitor_gramatical(ouro[T2]):", monitor_gramatical(CONSULTAS[5][2]))
print("monitor_gramatical(malformada[0]):", monitor_gramatical(consultas_malformadas()[0][0]))
print("checa_aninhamento(desbalanceada[0]):", checa_aninhamento(cadeias_desbalanceadas()[0]))

monitor_gramatical(ouro[T2]): True
monitor_gramatical(malformada[0]): False
checa_aninhamento(desbalanceada[0]): False


## `03_interpreter` — interpretador de referência

Define a semântica do subconjunto sobre os documentos (sem Elasticsearch). A
distinção crucial: uma cláusula `nested` é avaliada no escopo de **um mesmo item**
(correlação); uma condição sobre campo de item **sem** `nested` é **achatada** —
satisfeita se *qualquer* item a satisfaz, independentemente. `achatar` é a
transformação determinística que simula a falha de achatamento de C0.

In [4]:
def _cmp_range(valor, bounds):
    for op, alvo in bounds.items():
        v, a = float(valor), float(alvo)
        if op == "gt"  and not v >  a: return False
        if op == "gte" and not v >= a: return False
        if op == "lt"  and not v <  a: return False
        if op == "lte" and not v <= a: return False
    return True

def _campo_de_item(campo): return campo.startswith("det.")

def _item_satisfaz(clause, item):
    """Cláusula no escopo de UM item (semântica nested correlacionada)."""
    if "term" in clause:
        (campo, val), = clause["term"].items()
        return str(item.get(campo)) == str(val)
    if "range" in clause:
        (campo, bnd), = clause["range"].items()
        return campo in item and _cmp_range(item[campo], bnd)
    if "bool" in clause:
        b = clause["bool"]
        ok = all(_item_satisfaz(s, item) for s in b.get("must", []) + b.get("filter", []))
        if "should" in b:    ok = ok and any(_item_satisfaz(s, item) for s in b["should"])
        if "must_not" in b:  ok = ok and not any(_item_satisfaz(s, item) for s in b["must_not"])
        return ok
    return True

def _nota_satisfaz(clause, doc):
    """Cláusula no escopo da NOTA (nested = um mesmo item; achatada = qualquer item)."""
    if "nested" in clause:
        sub = clause["nested"]["query"]
        return any(_item_satisfaz(sub, it) for it in doc["itens"])
    if "term" in clause:
        (campo, val), = clause["term"].items()
        if _campo_de_item(campo):
            return any(str(it.get(campo)) == str(val) for it in doc["itens"])
        return str(doc["header"].get(campo)) == str(val)
    if "range" in clause:
        (campo, bnd), = clause["range"].items()
        if _campo_de_item(campo):
            return any(campo in it and _cmp_range(it[campo], bnd) for it in doc["itens"])
        return campo in doc["header"] and _cmp_range(doc["header"][campo], bnd)
    if "bool" in clause:
        b = clause["bool"]
        ok = all(_nota_satisfaz(s, doc) for s in b.get("must", []) + b.get("filter", []))
        if "should" in b:    ok = ok and any(_nota_satisfaz(s, doc) for s in b["should"])
        if "must_not" in b:  ok = ok and not any(_nota_satisfaz(s, doc) for s in b["must_not"])
        return ok
    return True

def executar(dsl, docs):
    """Retorna o conjunto de ids de notas casadas pela consulta."""
    raiz = dsl.get("query", dsl)
    return {d["id"] for d in docs if _nota_satisfaz(raiz, d)}

def achatar(dsl):
    """ACHATAMENTO: remove `nested`, promovendo a subconsulta ao nível da nota —
    troca a correlação 'no mesmo item' por 'em qualquer item'."""
    def t(c):
        if not isinstance(c, dict): return c
        if "nested" in c: return t(c["nested"]["query"])
        if "bool" in c:
            return {"bool": {k: [t(s) for s in v] for k, v in c["bool"].items()}}
        return c
    return {"query": t(dsl.get("query", dsl))}

def monitor_afd(traco, k=3):
    """Monitor regular do laço. Ações: gen, parse_ok, parse_fail, exec, repair.
    R1: toda 'exec' é precedida imediatamente por 'parse_ok'. R2: nº de 'repair' <= k."""
    anterior, repairs = None, 0
    for ev in traco:
        if ev == "exec" and anterior != "parse_ok": return False   # viola R1
        if ev == "repair":
            repairs += 1
            if repairs > k: return False                            # viola R2
        anterior = ev
    return True

print("executar(T2 ouro) sobre sintéticas:", sorted(executar(CONSULTAS[5][2], docs_t2)))

executar(T2 ouro) sobre sintéticas: ['sint_pos_1', 'sint_pos_2']


## `04_run` — execução determinística (sem LLM)

Calcula o resultado-ouro de cada consulta; aplica o monitor gramatical ao conjunto
ouro (aceita) e a um conjunto malformado (rejeita); para T2, constrói a variante
**achatada** e executa ambas; roda o monitor AFD sobre traços de ação. Os `assert`
travam as propriedades esperadas.

In [5]:
registros = []
for (pergunta, nivel, dsl, docs) in CONSULTAS:
    ids_ouro = executar(dsl, docs)
    dsl_achat = achatar(dsl)
    ids_achat = executar(dsl_achat, docs)
    registros.append({
        "pergunta": pergunta, "nivel": nivel, "dsl": dsl, "dsl_achat": dsl_achat,
        "parse_ouro": monitor_gramatical(dsl),
        "contextual_ok": monitor_contextual(dsl, CAMPOS_NESTED),
        "ids_ouro": ids_ouro, "ids_achat": ids_achat,
        "ex_achat": int(ids_achat == ids_ouro),          # EX por consulta (0/1)
        "falsos_pos": ids_achat - ids_ouro,
    })

# (a) Monitor gramatical sobre consultas malformadas (cada uma viola UMA regra da GLC)
malformadas = consultas_malformadas()
parse_malf = [monitor_gramatical(s) for s, _motivo in malformadas]

# (b) Monitor de pilha visível (VPA): aceita aninhamento balanceado, rejeita desbalanceado
aninh_ouro   = [checa_aninhamento(json.dumps(r["dsl"])) for r in registros]
desbal       = cadeias_desbalanceadas()
aninh_desbal = [checa_aninhamento(s) for s in desbal]

# (c) Monitor contextual: aceita path='det' (nested), rejeita path que não é campo nested
contextual_gold = [monitor_contextual(r["dsl"], CAMPOS_NESTED) for r in registros]
consulta_path_ruim = {"query": {"nested": {"path": "emit",
                       "query": {"term": {"det.prod.CFOP": "6108"}}}}}
contextual_neg = monitor_contextual(consulta_path_ruim, CAMPOS_NESTED)

# (d) Monitor AFD do laço (R1, R2)
tracos = {
    "valido":        (["gen", "parse_fail", "repair", "parse_ok", "exec"], True),
    "exec_sem_parse":(["gen", "exec"], False),                       # viola R1
    "parse_falho":   (["gen", "parse_fail", "exec"], False),         # viola R1
    "excesso_reparo":(["gen"] + ["parse_fail", "repair"] * 4, False),# viola R2 (k=3)
}
afd = {nome: monitor_afd(tr, k=3) for nome, (tr, _) in tracos.items()}

# ---- Asserts (travam as propriedades demonstradas) ----
assert all(r["parse_ouro"] for r in registros), "consulta-ouro rejeitada!"
assert not any(parse_malf), "consulta malformada aceita pelo parser!"
assert all(aninh_ouro), "VPA: aninhamento-ouro rejeitado!"
assert not any(aninh_desbal), "VPA: cadeia desbalanceada aceita!"
assert all(contextual_gold), "contextual: path nested válido rejeitado!"
assert contextual_neg is False, "contextual: path NÃO-nested deveria ser rejeitado!"
for r in registros:
    if r["nivel"] in ("T0", "T1"):
        assert r["ids_achat"] == r["ids_ouro"], f"achatar alterou {r['nivel']}!"
    if r["nivel"] == "T2":
        assert r["ids_achat"] != r["ids_ouro"], "T2: achatar deveria divergir!"
        assert len(r["falsos_pos"]) > 0, "T2: esperava falso positivo!"
for nome, (tr, esperado) in tracos.items():
    assert afd[nome] is esperado, f"AFD divergiu em {nome}"
print("*** TODOS OS ASSERTS PASSARAM (sem LLM, sem rede) ***")
print("VPA: ouro balanceado", aninh_ouro.count(True), "/", len(aninh_ouro),
      "| desbalanceadas rejeitadas", aninh_desbal.count(False), "/", len(aninh_desbal))
print("Contextual: path='det' aceito; path='emit' rejeitado =", contextual_neg is False)
print("Falsos positivos T2 (1a consulta):", sorted(registros[5]["falsos_pos"]))

*** TODOS OS ASSERTS PASSARAM (sem LLM, sem rede) ***
VPA: ouro balanceado 7 / 7 | desbalanceadas rejeitadas 3 / 3
Contextual: path='det' aceito; path='emit' rejeitado = True
Falsos positivos T2 (1a consulta): ['sint_trap_1', 'sint_trap_2', 'sint_trap_3']


## `05_metrics` — métricas reprodutíveis

As métricas **estruturais** são calculadas a partir dos registros determinísticos.
As métricas que dependem de LLM real (EX de C0/C2 gerados, iterações-até-válida)
ficam marcadas como **N/A nesta fase**.

In [6]:
from statistics import mean

niveis = ["T0", "T1", "T2"]
parse_rate_ouro = mean(int(r["parse_ouro"]) for r in registros)
parse_rate_malf = mean(int(p) for p in parse_malf)
ex_achat_nivel = {n: mean(r["ex_achat"] for r in registros if r["nivel"] == n) for n in niveis}

# Falso positivo de T2 sobre as notas sintéticas (1a consulta T2 como canônica)
r_t2 = next(r for r in registros if r["nivel"] == "T2")
nao_deviam = {d["id"] for d in docs_t2} - r_t2["ids_ouro"]   # notas que não deveriam casar
fp = r_t2["ids_achat"] - r_t2["ids_ouro"]
taxa_fp = len(fp) / len(nao_deviam)

# Nested-Correctness determinística: usa `nested` onde o nível (T1/T2) exige.
def usa_nested(dsl):
    def tem(c):
        if not isinstance(c, dict): return False
        if "nested" in c: return True
        if "bool" in c: return any(tem(s) for lst in c["bool"].values() for s in lst)
        return False
    return tem(dsl.get("query", dsl))
exige_nested = [r for r in registros if r["nivel"] in ("T1", "T2")]
nested_corr_ouro  = mean(int(usa_nested(r["dsl"]))       for r in exige_nested)
nested_corr_achat = mean(int(usa_nested(r["dsl_achat"])) for r in exige_nested)

metricas = {
    "parse_rate_ouro": parse_rate_ouro, "parse_rate_malformadas": parse_rate_malf,
    "ex_ouro": 1.0, "ex_achatada_por_nivel": ex_achat_nivel,
    "nested_correctness_ouro": nested_corr_ouro, "nested_correctness_achatada": nested_corr_achat,
    "t2_nested_casa": sorted(r_t2["ids_ouro"]), "t2_achatada_casa": sorted(r_t2["ids_achat"]),
    "t2_falsos_positivos": sorted(fp), "t2_taxa_fp": taxa_fp,
    "vpa_aninhamento_ok": all(aninh_ouro) and not any(aninh_desbal),
    "contextual_rejeita_path_invalido": contextual_neg is False,
    "afd": afd,
    # Métricas que exigem LLM real — NÃO inventar; ficam para a fase posterior.
    "nested_correctness_llm": "N/A (consultas geradas — fase posterior)",
    "ex_c0_llm": "N/A (fase posterior)", "ex_c2_llm": "N/A (fase posterior)",
    "iteracoes_ate_valida": "N/A (fase posterior)",
}
print(f"Parse Rate  | ouro = {parse_rate_ouro:.0%} | malformadas = {parse_rate_malf:.0%}")
print(f"EX(ouro) = {metricas['ex_ouro']:.0%}")
print("EX(achatada) por nível:", {k: f'{v:.0%}' for k, v in ex_achat_nivel.items()})
print(f"Nested-Correctness | ouro = {nested_corr_ouro:.0%} | achatada = {nested_corr_achat:.0%}"
      "  (valor de consultas geradas por LLM: N/A nesta fase)")
print(f"VPA aninhamento (aceita ouro / rejeita desbalanceada): {metricas['vpa_aninhamento_ok']}")
print(f"Contextual rejeita path inválido: {metricas['contextual_rejeita_path_invalido']}")
print(f"T2: nested casa {metricas['t2_nested_casa']}")
print(f"T2: achatada casa {metricas['t2_achatada_casa']}")
print(f"T2: FALSOS POSITIVOS {metricas['t2_falsos_positivos']} | taxa = {taxa_fp:.0%}")
print("Monitor AFD (R1/R2):", afd)
print("LLM (C0/C2 EX, iterações): N/A nesta fase determinística")

Parse Rate  | ouro = 100% | malformadas = 0%
EX(ouro) = 100%
EX(achatada) por nível: {'T0': '100%', 'T1': '100%', 'T2': '0%'}
Nested-Correctness | ouro = 100% | achatada = 0%  (valor de consultas geradas por LLM: N/A nesta fase)
VPA aninhamento (aceita ouro / rejeita desbalanceada): True
Contextual rejeita path inválido: True
T2: nested casa ['sint_pos_1', 'sint_pos_2']
T2: achatada casa ['sint_pos_1', 'sint_pos_2', 'sint_trap_1', 'sint_trap_2', 'sint_trap_3']
T2: FALSOS POSITIVOS ['sint_trap_1', 'sint_trap_2', 'sint_trap_3'] | taxa = 75%
Monitor AFD (R1/R2): {'valido': True, 'exec_sem_parse': False, 'parse_falho': False, 'excesso_reparo': False}
LLM (C0/C2 EX, iterações): N/A nesta fase determinística


## `06_figuras` — figuras (salvas em `figs/`)

Gera as figuras que ilustram o Capítulo 4: EX por nível (achatada × ouro) e o
falso positivo de T2 (achatada × nested). Reprodutíveis e sem números inventados.

In [7]:
os.makedirs("figs", exist_ok=True)

# Fig A — EX(achatada) por nível (a perda do achatamento concentra-se em T2)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(niveis, [ex_achat_nivel[n] for n in niveis], color=["#4c72b0"]*2 + ["#c44e52"])
ax.axhline(1.0, ls="--", c="gray", lw=1, label="EX(ouro) = 100%")
ax.set_ylim(0, 1.08); ax.set_ylabel("EX(achatada)"); ax.set_title("EX por nível — achatada × ouro")
ax.legend(); fig.tight_layout(); fig.savefig("figs/ex_por_nivel.png", dpi=150); plt.close(fig)

# Fig B — Falso positivo de T2 (nº de notas casadas: achatada × nested)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(["nested (ouro)", "achatada"],
       [len(r_t2["ids_ouro"]), len(r_t2["ids_achat"])], color=["#55a868", "#c44e52"])
ax.set_ylabel("nº de notas casadas (de %d)" % len(docs_t2))
ax.set_title("T2: falso positivo do achatamento\n(%d falsos positivos)" % len(fp))
fig.tight_layout(); fig.savefig("figs/t2_falso_positivo.png", dpi=150); plt.close(fig)

# Fig C — Parse Rate (ouro × malformadas)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(["ouro", "malformadas"], [parse_rate_ouro, parse_rate_malf], color=["#55a868", "#c44e52"])
ax.set_ylim(0, 1.08); ax.set_ylabel("Parse Rate"); ax.set_title("Monitor gramatical: aceita × rejeita")
fig.tight_layout(); fig.savefig("figs/parse_rate.png", dpi=150); plt.close(fig)

print("Figuras salvas em figs/:", sorted(os.listdir("figs")))

Figuras salvas em figs/: ['ex_por_nivel.png', 'parse_rate.png', 't2_falso_positivo.png']
